In [72]:
%pip install pandas numpy plotly ipywidgets nbformat


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. Introduction

### 1.1. Dataset

This notebook is to explore data visualization of Spotify 2023 dataset available on Kaggle. Here is the dataset description based on the website: <br><br> "This dataset contains a comprehensive list of the most famous songs of 2023 as listed on Spotify. The dataset offers a wealth of features beyond what is typically available in similar datasets. It provides insights into each song's attributes, popularity, and presence on various music platforms. The dataset includes information such as track name, artist(s) name, release date, Spotify playlists and charts, streaming statistics, Apple Music presence, Deezer presence, Shazam charts, and various audio features."<br><br>This dataset can be found at https://www.kaggle.com/datasets/nelgiriyewithana/top-spotify-songs-2023

### 1.2. Visualization Technique

There are a lot of songs released in previous years, yet still standing in top 1000 most streamed songs in 2023, out of over 100 million songs offered in Spotify. We think filtering data based on released year will provide interestesting insights of how songs stayed active streaming, given how fast-changing the industry is. For this purpose, our dashboard will include a drop down menu, allows user to select a released year. Charts on dashboard are filtered based on that specific released year. With this interactive feature, we can observe if any patterns stay consistent over the years, or things are also fast-changing as the industry.

For this dashboard, we are using four plots in total: 

- Bar chart for top 10 songs released that year. 
- Scatter plot shows relationship between stream counts of a song, and the number of Spotify playlists the song was in.
- Histogram shows count of top songs released each month of that year.
- Violin plot shows distribution of danceability, valence, and energy.

One of the first questions we have is "What are most streamed songs released that year? And what were their rankings?" Bar chart is the best to visualize this. Horizontal bar chart is ideal for long labels like songs' titles. However, bar chart is not the best at showing relationship between two variables. This is where scatter plot shines. We use scatter plot to see if being in more Spotify playlists associates with higher stream count of the same song. Next, we are curious if any specific released month associates with higher number of top songs over the years. A histogram will clearly visualize number of top songs per each released month. Observing this histogram over several released years will yield some insights. Last but not least, we'd like to see audio feature distribution of all the top songs released in the year. Since danceability, valence, and energy percentage share the same range of values (0-100), we can use violin plot for all three with the same y-axis. 

### 1.3. Visualization Library

In this dashboard, we use Plotly library for our visualization. Plotly is created by [Plotly](plotly.com), a technical computing company headquartered in Montreal, Canada ([source](https://en.wikipedia.org/wiki/Plotly)). There were four founders at Plotly: Alex Johnson, Jack Parmer, Chris Parmer, and Matthew Sundquist. There were also two early employees at Plotly: Christophe Viau, and Ben Postlethwaite. Their background spanned across science, energy, and data analysis and visualization. Plotly is an open-source, declarative visualization library. 

To install Plotly, you can simply execute below command:
```
pip install plotly
```
Altenatively, you can install it via conda:
```
conda install -c plotly plotly
```

The reason we selected Plotly and ipywidgets for this projects are:

- Declarative library: we can easily create our charts in simple line of code with Plotly, without worrying too much about layout details (axis, labels...)
- Seamless integration with Jupyter Notebook.
- Built-in interactive features: with minimal code, Plotly chart reveals further data point details as user hovers over the charts.
- Full support on all our planned chart types and interactive features.
- Ease of dashboarding: Plotly and ipywidgets allow us to create a functioning, interactive dashboard inside Jupyter Notebook environment without needing extra web application overhead.

For our small dataset, this framework works fine. However, for a large dataset, this framework will show its limitations:

- Memory and performance limit: Rendering interactive charts for a large dataset is much slower than a static chart.
- Widget interactivity limit: ipywidgets requires an active, running Python kernel to handle event triggers (e.g: our dropdown menu). If the notebook is exported as a static HTML page, the interactive dropdown won't function unless pre-rendered or connected to a live kernel.

## 2. Preparation: Reading and Cleaning Data

### 2.1. Libraries and Data Import

In this dashboard, we only need 3 libraries:

- pandas to read data file, then analyze it.
- plotly to plot interactive charts.
- ipywidget for the dropdown menu.

These three libraries will work seamlessly in our Jupyter Notebook environment to render an interactive dashboard at the end of this notebook.

Let's import the three libraries.

In [ ]:
import pandas as pd
import plotly.express as px
from ipywidgets import interact, interactive, Dropdown

Now we are ready to read the data file into a dataframe called "df".

In [74]:
df = pd.read_csv('spotify-2023.csv', encoding='cp1252')
df.head()

,track_name,artist(s)_name,artist_count,released_year,released_month,released_day,in_spotify_playlists,in_spotify_charts,streams,in_apple_playlists,...,bpm,key,mode,danceability_%,valence_%,energy_%,acousticness_%,instrumentalness_%,liveness_%,speechiness_%
0,Seven (feat. Latto) (Explicit Ver.),"Latto, Jung Kook",2,2023,7,14,553,147,141381703,43,...,125,B,Major,80,89,83,31,0,8,4
1,LALA,Myke Towers,1,2023,3,23,1474,48,133716286,48,...,92,C#,Major,71,61,74,7,0,10,4
2,vampire,Olivia Rodrigo,1,2023,6,30,1397,113,140003974,94,...,138,F,Major,51,32,53,17,0,31,6
3,Cruel Summer,Taylor Swift,1,2019,8,23,7858,100,800840817,116,...,170,A,Major,55,58,72,11,0,11,15
4,WHERE SHE GOES,Bad Bunny,1,2023,5,18,3133,50,303236322,84,...,144,A,Minor,65,23,80,14,63,11,6


Let's take a look at the dataframe shape.

In [75]:
df.shape

(953, 24)

Our dataset has only 953 rows. Small enough for this mini-project and will not cause problem rendering later. 24 columns provide meaningful features to potential yield interesting insights in the Analyze phase.

### 2.2. Data Cleaning

Please note that this dataset contains special characters in track_name and artist(s)_name. We attempted to clean them with various encoding parameters in reading csv function, and accent removal. However, these attempts did not help since the names were hard coded with special characters at the source. The only way to fix the typo and remove accent is to manually edit special characters one by one. Due to time limit, we decided to leave the names as they are. Please expect to see special characters appear occasionally in track_name as the charts are shown.

#### 2.2.1. Check Value Range

Let's take a look see if numerical values have a reasonable range.

In [76]:
df.describe()

,artist_count,released_year,released_month,released_day,in_spotify_playlists,in_spotify_charts,in_apple_playlists,in_apple_charts,in_deezer_charts,bpm,danceability_%,valence_%,energy_%,acousticness_%,instrumentalness_%,liveness_%,speechiness_%
count,953.000000,953.000000,953.000000,953.000000,953.000000,953.000000,953.000000,953.000000,953.000000,953.000000,953.00000,953.000000,953.000000,953.000000,953.000000,953.000000,953.000000
mean,1.556139,2018.238195,6.033578,13.930745,5200.124869,12.009444,67.812172,51.908709,2.666317,122.540399,66.96957,51.431270,64.279119,27.057712,1.581322,18.213012,10.131165
std,0.893044,11.116218,3.566435,9.201949,7897.608990,19.575992,86.441493,50.630241,6.035599,28.057802,14.63061,23.480632,16.550526,25.996077,8.409800,13.711223,9.912888
min,1.000000,1930.000000,1.000000,1.000000,31.000000,0.000000,0.000000,0.000000,0.000000,65.000000,23.00000,4.000000,9.000000,0.000000,0.000000,3.000000,2.000000
25%,1.000000,2020.000000,3.000000,6.000000,875.000000,0.000000,13.000000,7.000000,0.000000,100.000000,57.00000,32.000000,53.000000,6.000000,0.000000,10.000000,4.000000
50%,1.000000,2022.000000,6.000000,13.000000,2224.000000,3.000000,34.000000,38.000000,0.000000,121.000000,69.00000,51.000000,66.000000,18.000000,0.000000,12.000000,6.000000
75%,2.000000,2022.000000,9.000000,22.000000,5542.000000,16.000000,88.000000,87.000000,2.000000,140.000000,78.00000,70.000000,77.000000,43.000000,0.000000,24.000000,11.000000
max,8.000000,2023.000000,12.000000,31.000000,52898.000000,147.000000,672.000000,275.000000,58.000000,206.000000,96.00000,97.000000,97.000000,97.000000,91.000000,97.000000,64.000000


#### 2.2.2. Check Missing Values

Those ranges look good. Let's see if this dataset has any missing values.

In [77]:
df.isna().any()

track_name              False
artist(s)_name          False
artist_count            False
released_year           False
released_month          False
released_day            False
in_spotify_playlists    False
in_spotify_charts       False
streams                 False
in_apple_playlists      False
in_apple_charts         False
in_deezer_playlists     False
in_deezer_charts        False
in_shazam_charts         True
bpm                     False
key                      True
mode                    False
danceability_%          False
valence_%               False
energy_%                False
acousticness_%          False
instrumentalness_%      False
liveness_%              False
speechiness_%           False
dtype: bool

In [78]:
df.isna().sum().sum()

np.int64(145)

The 'in_shazam_charts' and 'key' columns have total of 145 missing values. We may not use these features anyway so let's drop them off the dataset.

In [79]:
df = df.drop(columns=['in_shazam_charts', 'key'])

Let's make sure these columns are dropped.

In [80]:
df.columns

Index(['track_name', 'artist(s)_name', 'artist_count', 'released_year',
       'released_month', 'released_day', 'in_spotify_playlists',
       'in_spotify_charts', 'streams', 'in_apple_playlists', 'in_apple_charts',
       'in_deezer_playlists', 'in_deezer_charts', 'bpm', 'mode',
       'danceability_%', 'valence_%', 'energy_%', 'acousticness_%',
       'instrumentalness_%', 'liveness_%', 'speechiness_%'],
      dtype='str')

#### 2.2.3. Check Data Types

Let's take a look at data type of each column.

In [81]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 953 entries, 0 to 952
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   track_name            953 non-null    str  
 1   artist(s)_name        953 non-null    str  
 2   artist_count          953 non-null    int64
 3   released_year         953 non-null    int64
 4   released_month        953 non-null    int64
 5   released_day          953 non-null    int64
 6   in_spotify_playlists  953 non-null    int64
 7   in_spotify_charts     953 non-null    int64
 8   streams               953 non-null    str  
 9   in_apple_playlists    953 non-null    int64
 10  in_apple_charts       953 non-null    int64
 11  in_deezer_playlists   953 non-null    str  
 12  in_deezer_charts      953 non-null    int64
 13  bpm                   953 non-null    int64
 14  mode                  953 non-null    str  
 15  danceability_%        953 non-null    int64
 16  valence_%          

We noticed that 'streams' column has string values, instead of integer. However, when converting these to integer, a value error was raised. Let's drop the row with value error.

In [82]:
# drop row with wrong stream count value
df = df[df['streams'] != 'BPM110KeyAModeMajorDanceability53Valence75Energy69Acousticness7Instrumentalness0Liveness17Speechiness3']

Now we can convert 'streams' values to int.

In [83]:
# Convert 'streams' column from string to int value
df['streams'] = df['streams'].astype('int')

Let's double check one more time.

In [84]:
df.info()

<class 'pandas.DataFrame'>
Index: 952 entries, 0 to 952
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   track_name            952 non-null    str  
 1   artist(s)_name        952 non-null    str  
 2   artist_count          952 non-null    int64
 3   released_year         952 non-null    int64
 4   released_month        952 non-null    int64
 5   released_day          952 non-null    int64
 6   in_spotify_playlists  952 non-null    int64
 7   in_spotify_charts     952 non-null    int64
 8   streams               952 non-null    int64
 9   in_apple_playlists    952 non-null    int64
 10  in_apple_charts       952 non-null    int64
 11  in_deezer_playlists   952 non-null    str  
 12  in_deezer_charts      952 non-null    int64
 13  bpm                   952 non-null    int64
 14  mode                  952 non-null    str  
 15  danceability_%        952 non-null    int64
 16  valence_%             95

Let's double check value ranges too.

In [105]:
df.describe()

,artist_count,released_year,released_month,released_day,in_spotify_playlists,in_spotify_charts,streams,in_apple_playlists,in_apple_charts,in_deezer_charts,bpm,danceability_%,valence_%,energy_%,acousticness_%,instrumentalness_%,liveness_%,speechiness_%
count,952.000000,952.000000,952.000000,952.000000,952.000000,952.000000,9.520000e+02,952.000000,952.000000,952.000000,952.000000,952.000000,952.000000,952.000000,952.000000,952.000000,952.000000,952.000000
mean,1.556723,2018.288866,6.038866,13.944328,5202.565126,12.022059,5.141374e+08,67.866597,51.963235,2.669118,122.553571,66.984244,51.406513,64.274160,27.078782,1.582983,18.214286,10.138655
std,0.893331,11.011397,3.564571,9.197223,7901.400683,19.582405,5.668569e+08,86.470591,50.628850,6.038152,28.069601,14.631282,23.480526,16.558517,26.001599,8.414064,13.718374,9.915399
min,1.000000,1930.000000,1.000000,1.000000,31.000000,0.000000,2.762000e+03,0.000000,0.000000,0.000000,65.000000,23.000000,4.000000,9.000000,0.000000,0.000000,3.000000,2.000000
25%,1.000000,2020.000000,3.000000,6.000000,874.500000,0.000000,1.416362e+08,13.000000,7.000000,0.000000,99.750000,57.000000,32.000000,53.000000,6.000000,0.000000,10.000000,4.000000
50%,1.000000,2022.000000,6.000000,13.000000,2216.500000,3.000000,2.905309e+08,34.000000,38.500000,0.000000,121.000000,69.000000,51.000000,66.000000,18.000000,0.000000,12.000000,6.000000
75%,2.000000,2022.000000,9.000000,22.000000,5573.750000,16.000000,6.738690e+08,88.000000,87.000000,2.000000,140.250000,78.000000,70.000000,77.000000,43.000000,0.000000,24.000000,11.000000
max,8.000000,2023.000000,12.000000,31.000000,52898.000000,147.000000,3.703895e+09,672.000000,275.000000,58.000000,206.000000,96.000000,97.000000,97.000000,97.000000,91.000000,97.000000,64.000000


All of our features of interest are now in correct type and reasonable range. We are ready to analyze our data.

## 3. Data Analysis and Visualization

### 3.1. What released years should we look into?

Let's take a look at released year of these top songs.

In [85]:
# check released_year unique values
print(df['released_year'].unique())

[2023 2019 2022 2013 2014 2018 2017 2020 2016 2012 1999 2008 1975 2021
 2015 2011 2004 1985 2007 2002 2010 1983 1992 1968 1984 2000 1997 1995
 2003 1973 1930 1994 1958 1957 1963 1959 1970 1971 1952 1946 1979 1950
 1942 1986 2005 1991 1996 1998 1982 1987]


In [86]:
len(df['released_year'].unique())

50

Interesting. There are songs released way back in 1930 yet still remain most streamed in 2023! Songs could stay timeless, sort of. It could be a nice idea to look into top songs break down by released year. However, it could be too much to analyze all 50 values of released year. We also need to make sure there are enough data points in the released year of interest.<br>Let's see how many data points in each released year.

In [87]:
with pd.option_context('display.max_rows', None):
    display(df['released_year'].value_counts())

released_year
2022    402
2023    175
2021    119
2020     37
2019     36
2017     23
2016     18
2013     13
2014     13
2015     11
2018     10
2012     10
2011     10
2010      7
2002      6
1999      5
2004      4
1984      4
2000      4
1958      3
1963      3
2008      2
1975      2
1985      2
1995      2
2003      2
1957      2
1959      2
1986      2
1991      2
1982      2
2007      1
1983      1
1992      1
1968      1
1997      1
1973      1
1930      1
1994      1
1970      1
1971      1
1952      1
1946      1
1979      1
1950      1
1942      1
2005      1
1996      1
1998      1
1987      1
Name: count, dtype: int64

Let's plot a histogram to visualize these value counts.

In [88]:
fig = px.histogram(
    df,
    x = 'released_year',
    title = 'Number of top songs based on released year'
)
fig.show()

Released year between **2011 to 2023** yields 10 or more rows of data. We will explore data visualization in each of these released years. 

Let's explore the features of this dataset again.

In [89]:
print(df.columns)

Index(['track_name', 'artist(s)_name', 'artist_count', 'released_year',
       'released_month', 'released_day', 'in_spotify_playlists',
       'in_spotify_charts', 'streams', 'in_apple_playlists', 'in_apple_charts',
       'in_deezer_playlists', 'in_deezer_charts', 'bpm', 'mode',
       'danceability_%', 'valence_%', 'energy_%', 'acousticness_%',
       'instrumentalness_%', 'liveness_%', 'speechiness_%'],
      dtype='str')


### 3.2. Bar chart: Top 10 Songs in Selected Released Year

First thing first, what are top ten songs in our interested released year? We may want to rank them in stream counts too. A bar chart is a great option for ranking visualization. For long axis label such as song's title, a horizontal bar chart will make long labels easier to read. You can easily create a bar chart in plotly by its bar() function (details below). To make our bar chart horizontal, simply set orientation='h'.

In [90]:
def top_stream_counts(df, year):
    this_year_df = df[df['released_year']==year][['track_name', 'artist(s)_name', 'streams']]
    this_year_sorted = this_year_df.sort_values(by='streams', ascending=False)
    top10 = this_year_sorted.head(10)
    top10_sorted = top10.sort_values(by='streams', ascending=True)

    fig = px.bar(
        top10_sorted,
        y = 'track_name',
        x = 'streams',
        hover_data='artist(s)_name',
        title = 'Top 10 most streamed songs released in ' + str(year),
        text_auto = True,
        orientation = 'h'
    )
    fig.update_layout(
        height=700, # give each bar enough thickness
        bargap=0.2, 
        yaxis_title="",
        xaxis_title="Stream Count",
        template='plotly_white',
        margin=dict(l=10, r=20, t=50, b=40)
    )
    #fig.show()
    return fig

Please feel free to modify the year parameter below to explore top 10 most streamed songs released between 2011-2023.

In [91]:
# let's test our bar chart
top_stream_counts(df, 2011).show()

### 3.3. Scatter plot: Is Being in More Spotify Playlists Associated With Higher Stream Counts?

Let's visualize the relationship between 'in_spotify_playlists' and 'streams' with a scatter plot. Scatter plot is one of the best options to visualize relationship between two variables. In addition, plotly scatterplot allows you to take extra variables into consideration by including them in size, color... paramaters. In this chart, we set opacity=0.7 so we can see more data points. We also set size='in_spotify_charts' see if being featured in more Spotify charts helps drive that song's stream count.

In [92]:
def stream_vs_playlist(df, year):
    this_year_df = df[df['released_year']==year][['track_name', 'artist(s)_name', 'streams', 'in_spotify_charts', 'in_spotify_playlists']]
    
    fig = px.scatter(
            this_year_df, 
            x='in_spotify_playlists', 
            y='streams',
            opacity=0.7,                 
            size='in_spotify_charts',                       
            hover_name='track_name',          
            hover_data=['artist(s)_name', 'in_spotify_charts'],
            title='Streams vs. Number of Spotify playlists of top songs released in ' + str(year)
        )
    
    fig.update_layout(template='plotly_white')
    #fig.show()
    return fig

Let's test it.

In [123]:
stream_vs_playlist(df, 2023).show()

Generally, the charts show a positive correlation between 'in_spotify_playlists' and 'streams'. However, it is not clear if being featured in more Spotify charts helps drive the song's stream count. It is especially unclear for songs released between 2011-2017.

If you select released year of 2022 or 2023, you may catch the limitation of scatter plot: when there are too many data points, it may be harder to observe the trend or relationship between variables, since too much scattering points cloud the trend. Too many consideration (size, color, symbol...) makes it even more confusing. It is harder for viewer to capture the core ideas of your chart within 5 seconds.

### 3.4. Histogram: How Many Songs Stayed Top Streamed in Each Month of the Selected Released Year?

Histogram is a natural option for this question. Below is how to create a histogram in plotly.

In [98]:
def released_month_hist(df, year):
    this_year_df = df[df['released_year']==year]['released_month']

    fig = px.histogram(
        this_year_df,
        x = 'released_month',
        nbins = 12,
        title = 'Number of most streamed songs released each month in ' + str(year),
        text_auto = True
    )
    fig.update_xaxes(
        dtick=1,                            
        tickvals=list(range(1, 13)),       
        ticktext=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']        
    )                           
    fig.update_layout(
        xaxis_title='Released month in ' + str(year),
        yaxis_title='Number of most streamed songs in 2023',
        bargap=0.2,
        template='plotly_white'
    )
    
    #fig.show()
    return fig

Let's test it. Again, please feel free to change the year parameter in our function.

In [99]:
released_month_hist(df, 2022).show()

As we select released year from 2011 to 2023, we can see that the 'top month' is not consistent. Between 2011 and 2015, January had the most released songs still stayed top streamed in 2023. However, starting 2016, there is no consistent month that holds this record.

### 3.5. Violin plot: Audio Features Distribution

In [100]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def dve_vplot(df, year):
    cols = ['danceability_%', 'valence_%', 'energy_%']
    this_year_df = df[df['released_year']==year][cols]

    fig = make_subplots(
        rows=1, 
        cols=3, 
        subplot_titles=['Danceability', 'Valence', 'Energy'],
        shared_yaxes=True
    )
    for i, col in enumerate(cols, start=1):
        fig.add_trace(
            go.Violin(
                y=this_year_df[col],
                name=col.replace('_%', ' percentage').capitalize(),
                box_visible=True,            
                spanmode='hard', # to prevent plots go beyond 100%                   
                showlegend=False
            ),
            row=1, col=i
        )
    fig.update_yaxes(range=[0, 105], title_text="Percentage (%)", row=1, col=1)
    fig.update_layout(
        title='Audio feature distributions of top songs released in ' + str(year),
        template='plotly_white',
        height=500,
        margin=dict(t=80, b=40, l=60, r=40)
    )
    
    #fig.show()
    return fig

In [101]:
dve_vplot(df, 2023).show()

In [102]:
def create_dashboard(df, year):
    ax1 = top_stream_counts(df, year)
    ax2 = released_month_hist(df, year)
    ax3 = stream_vs_playlist(df, year)
    ax4 = dve_vplot(df, year)

    titles = [
        ax1.layout.title.text if ax1.layout.title.text else "Top Stream Counts",
        ax2.layout.title.text if ax2.layout.title.text else "Streams vs Playlist",
        ax3.layout.title.text if ax3.layout.title.text else "Release Month Distribution",
        ax4.layout.title.text if ax4.layout.title.text else "Audio Features"
    ]

    dashboard = make_subplots(
        rows=2, 
        cols=2,
        subplot_titles=titles,
        horizontal_spacing=0.12,
        vertical_spacing=0.15
    )

    fig_map = [
        (ax1, 1, 1),
        (ax2, 1, 2),
        (ax3, 2, 1),
        (ax4, 2, 2)
    ]

    for fig, r, c in fig_map:
        for trace in fig.data:
            dashboard.add_trace(trace, row=r, col=c)

    # this is to fix x-label on histogram
    month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                   'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
                   
    dashboard.update_xaxes(
        dtick=1,
        tickvals=list(range(1, 13)),
        ticktext=month_names,
        range=[0.5, 12.5],
        row=1, col=2 # apply this update x-axis on row 1, col 2, aka, ax2 - histogram only
    )

    dashboard.update_layout(
        title_text=f"Spotify Music Dashboard ({year})",
        title_font_size=20,
        height=900,
        width=1700,
        template='plotly_white',
        showlegend=False,
        bargap=0.15,  # <--- Restores spacing between bars globally across subplots
        margin=dict(t=100, b=50, l=50, r=50)
    )

    dashboard.show()            

In [103]:
create_dashboard(df, 2022)

In [ ]:
import ipywidgets as widgets
from ipywidgets import interact

year_options = list(range(2011, 2024))

@interact(
    year=widgets.Dropdown(
        options=year_options,
        value=2023, # default option
        description='Released year: ',
        style={'description_width': 'initial'}
    )
)
def interactive_dashboard(year):
    # Calls your existing function with the selected year
    create_dashboard(df, year)

interactive(children=(Dropdown(description='Released year: ', index=12, options=(2011, 2012, 2013, 2014, 2015,…